# Verificacion de la calidad de la tabla plata.pagos

Proposito del script:  
- Verificar columna por columna la calidad de los datos.

# Estableciendo la Conexion

In [1]:
import pandas as pd
import numpy as np
from datetime import date 
from conexiones_y_rutas import obtener_engine
from funciones import recalcular_monto_interes_programado,recalcular_monto_mora_pagado, formato_estado_pago
engine = obtener_engine()


df_pagos = pd.read_sql(
    "SELECT * FROM plata.pagos",
    con=engine
)
df_pagos_tra = df_pagos.copy()

# Archivos de Ayuda

In [2]:
df_prestamos_tra =  pd.read_sql(
    "SELECT * FROM plata.prestamos",
    con=engine
)
df_prestamos_tra.head()

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga
0,1,4848,23,6,60,CONT-00000001,2023-05-07,2025-10-23,9218.73,3596.20,...,Vigente,0,Normal,n/a,0.0,Viaje / Turismo,Transferencia Bancaria,2023-06-06,2024-12-27,2026-08-03 18:46:43.170
1,2,44,5,1,21,CONT-00000002,2021-02-12,2025-07-21,7767.38,1473.62,...,Vigente,0,Normal,Aval,0.0,Mejoras del Hogar,Agencia,2021-03-14,2024-12-23,2026-08-03 18:46:43.170
2,3,2474,16,1,35,CONT-00000003,2023-11-22,2027-11-01,12512.04,10640.43,...,Vigente,0,Normal,Aval,0.0,Mejoras del Hogar,Transferencia Bancaria,2023-12-22,2024-12-16,2026-08-03 18:46:43.170
3,4,637,15,4,32,CONT-00000004,2024-04-19,2024-10-16,3555.11,0.00,...,Cancelado,16,CPP,Carta Fianza,0.0,Expansión de Negocio,Agencia,2024-05-19,2024-11-01,2026-08-03 18:46:43.170
4,5,3622,10,4,12,CONT-00000005,2020-04-19,2022-10-06,8388.15,0.00,...,Cancelado,0,Normal,Sin Garantía,0.0,Capital de Trabajo,Agencia,2020-05-19,2022-10-06,2026-08-03 18:46:43.170


In [3]:
columnas_fecha = [
    "fecha_otorgamiento",
    "fecha_vencimiento",
    "fecha_primer_pago_programado",
    "fecha_ultimo_pago_real"
]

for columna in columnas_fecha:
    df_prestamos_tra[columna] = pd.to_datetime(df_prestamos_tra[columna])

# Resumen de las Columnas

- **pago_id**: Identificador unico de cada pago.  
- **prestamo_id**: Identificador del prestamo asociado a dicho pago.  
- **numero_cuota**: Numero de cuota del pago.  
- **fecha_vencimiento_cuota**: Fecha de vencimiento del pago.  
- **fecha_pago**: Fecha real de pago.  
- **monto_cuota_programada**: Monto de cuota programada.  
- **monto_capital_programado**: Monto capital programado.  
- **monto_interes_programado**: Monto de interes programado.
- **monto_pagado_total**: Monto real pagado por el cliente.
- **monto_capital_pagado**: Parte del monto total que se destina a reducir la deuda (monto real que se descuenta del total de la deuda).  
- **monto_interes_pagado**: Parte del moto que se destina a pagar intereses (monto que se paga por el interes NO SE DESCUENTA DE LA DEUDA).  
- **monto_mora_pagado**: Monto que se paga por la mora, solo se genera si existe mora en el pago (NO SE DESCUENTA DE LA DEUDA)
- **saldo_capital_despues_pago**: Saldo de la deuda despues de restar el monto_capital_pagado. 
- **dias_retraso**: Dias de retraso (diferencia entre fecha_vencimiento_cuota y fecha_pago).
- **estado_pago**: Situacion del pago (ejem: Puntual o Tardío).
- **canal_pago**: Canal donde se realizo el pago (ejem: Agencia o App Móvi).
- **referencia_pago**: Codigo de referencia del pago formato REF00000000(PAGO_ID) => 13 caracteres.

# Verificacion de la Calidad de Datos

In [4]:
df_pagos_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134911 entries, 0 to 134910
Data columns (total 18 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   pago_id                     134911 non-null  int64         
 1   prestamo_id                 134911 non-null  int64         
 2   numero_cuota                134911 non-null  int64         
 3   fecha_vencimiento_cuota     134911 non-null  object        
 4   fecha_pago                  134911 non-null  object        
 5   monto_cuota_programada      134911 non-null  float64       
 6   monto_capital_programado    134911 non-null  float64       
 7   monto_interes_programado    134911 non-null  float64       
 8   monto_pagado_total          134911 non-null  float64       
 9   monto_capital_pagado        134911 non-null  float64       
 10  monto_interes_pagado        134911 non-null  float64       
 11  monto_mora_pagado           134911 non-

In [5]:
columnas_fecha = [
    "fecha_vencimiento_cuota",
    "fecha_pago"
]

for columna in columnas_fecha:
    df_pagos_tra[columna] = pd.to_datetime(df_pagos_tra[columna])

In [6]:
df_pagos_tra.head()

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga
0,1,1,1,2023-06-06,2023-06-06,393.63,238.91,154.72,393.63,238.91,154.72,0.0,8979.82,0,Puntual,APP Móvil,REF0000000001,2026-08-03 18:46:46.943333
1,2,1,2,2023-07-06,2023-07-06,393.63,242.92,150.71,393.63,242.92,150.71,0.0,8736.90,0,Puntual,Débito Automático,REF0000000002,2026-08-03 18:46:46.943333
2,3,1,3,2023-08-05,2023-08-05,393.63,247.00,146.63,393.63,247.00,146.63,0.0,8489.90,0,Puntual,Agencia,REF0000000003,2026-08-03 18:46:46.943333
3,4,1,4,2023-09-04,2023-09-04,393.63,251.14,142.49,393.63,251.14,142.49,0.0,8238.76,0,Puntual,Agencia,REF0000000004,2026-08-03 18:46:46.943333
4,5,1,5,2023-10-04,2023-10-04,393.63,255.36,138.27,393.63,255.36,138.27,0.0,7983.40,0,Puntual,Débito Automático,REF0000000005,2026-08-03 18:46:46.943333


In [7]:
# Verifica si existen registro duplicados 
# Resultados Esperados: Tabla Vacia
df_pagos_tra[df_pagos_tra.duplicated(keep=False)]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


## pago_id

In [8]:
# Verifica si existen ids negativos o 0
# Resultados Esperados: Tabla Vacia 
df_pagos_tra[df_pagos_tra.pago_id <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


In [9]:
# Verifica si existen identificadores de pagos duplicados 
# Resultados Esperados: Tabla Vacia 
df_pagos_tra[df_pagos_tra.pago_id.duplicated(keep=False)]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


## prestamo_id

In [10]:
# Verifica si todos los pagos tienen asociado un prestamo 
# Resultados Esperados: both: 134911, left_only: 0, right_only: 0
verificar_union = df_pagos_tra.merge(
    df_prestamos_tra,
    on='prestamo_id',
    how='left',
    indicator=True
)
verificar_union._merge.value_counts()

_merge
both          134911
left_only          0
right_only         0
Name: count, dtype: int64

## numero_cuota

In [11]:
# Verfica si existen valores negativos o 0 
# Resultados Esperados: Tabla Vacia 
df_pagos_tra[df_pagos_tra.numero_cuota <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


In [12]:
# Selecciona el maximo numero de cuota para cada prestamo_id 
group_pagos = df_pagos_tra.groupby('prestamo_id').agg(
    max_num_cuota = ('numero_cuota','max')
).reset_index()

# Left Join, tabla agrupada de pagos y prestamos 
merge_pagos_prestamos = group_pagos.merge(
    right= df_prestamos_tra,
    on='prestamo_id',
    how='left'
)

merge_pagos_prestamos.head()

,prestamo_id,max_num_cuota,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga
0,1,20,4848,23,6,60,CONT-00000001,2023-05-07,2025-10-23,9218.73,...,Vigente,0,Normal,n/a,0.0,Viaje / Turismo,Transferencia Bancaria,2023-06-06,2024-12-27,2026-08-03 18:46:43.170
1,2,47,44,5,1,21,CONT-00000002,2021-02-12,2025-07-21,7767.38,...,Vigente,0,Normal,Aval,0.0,Mejoras del Hogar,Agencia,2021-03-14,2024-12-23,2026-08-03 18:46:43.170
2,3,13,2474,16,1,35,CONT-00000003,2023-11-22,2027-11-01,12512.04,...,Vigente,0,Normal,Aval,0.0,Mejoras del Hogar,Transferencia Bancaria,2023-12-22,2024-12-16,2026-08-03 18:46:43.170
3,4,6,637,15,4,32,CONT-00000004,2024-04-19,2024-10-16,3555.11,...,Cancelado,16,CPP,Carta Fianza,0.0,Expansión de Negocio,Agencia,2024-05-19,2024-11-01,2026-08-03 18:46:43.170
4,5,30,3622,10,4,12,CONT-00000005,2020-04-19,2022-10-06,8388.15,...,Cancelado,0,Normal,Sin Garantía,0.0,Capital de Trabajo,Agencia,2020-05-19,2022-10-06,2026-08-03 18:46:43.170


In [13]:
# Verifica que el numero maximo de cuota por prestamo_id sea igual al numero de cuotas pagadas en la tabla limpio_prestamos
# Resultados Esperados: Tabla Vacia 
merge_pagos_prestamos[merge_pagos_prestamos.max_num_cuota != merge_pagos_prestamos.numero_cuotas_pagadas]

,prestamo_id,max_num_cuota,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


**NOTA:** Falta revisar que el numero de cuota sea secuencial, es decir si tenemos como maximo el numero de cuotas pagadas 20, existan cuotas del 1 al 20.  
Como existen ids duplicados, voy a realizar esta limpieza en:  
[Secuencialidad Numero Cuota](#secuencialidad-numero_cuota-v2)

## fecha_vencimiento_cuota

In [14]:
# Fecha que generar error al transformar a date time
# Resultados Esperados: Tabla Vacia 
error_fecha_vencimiento = pd.to_datetime(df_pagos_tra.fecha_vencimiento_cuota,errors='coerce')
df_pagos_tra.fecha_vencimiento_cuota[error_fecha_vencimiento.isna()].head()

Series([], Name: fecha_vencimiento_cuota, dtype: datetime64[ns])

## fecha_pago

In [15]:
# Fecha que generar error al transformar a date time
# Resultados Esperados: Tabla Vacia 
error_fecha_pago = pd.to_datetime(df_pagos_tra.fecha_pago,errors='coerce')
df_pagos_tra.fecha_pago[error_fecha_pago.isna()].head()

Series([], Name: fecha_pago, dtype: datetime64[ns])

## monto_cuota_programada

In [16]:
# Verifica si existen montos negativos o  0 
df_pagos_tra[df_pagos_tra.monto_cuota_programada <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


In [17]:
# Calcula el promedio, porque, en caso exista una cuota con otro valor el resultado no va a ser igual entonces tendria que cambiar los valores 
group_cuota_programada = df_pagos_tra.groupby('prestamo_id').agg(
    cuota_pro = ('monto_cuota_programada','mean')
).reset_index()

# Redondea hasta las decenas porque, es el formato que se utiliza en prestamos
group_cuota_programada['cuota_pro'] = group_cuota_programada.cuota_pro.apply(lambda x: round(x,2))

# Left Join, prestamo_id 
merge_cuota_progra = group_cuota_programada.merge(
    right=df_prestamos_tra,
    on='prestamo_id',
    how='left'
)

# Verifica si las cuotas programadas en pagos son diferentes a las cuotas programadas en prestamos 
# Resultados Esperados: Tabla Vacia
merge_cuota_progra[['prestamo_id','cuota_pro','cuota_programada']][
    merge_cuota_progra.cuota_pro != merge_cuota_progra.cuota_programada
]

,prestamo_id,cuota_pro,cuota_programada


## monto_capital_programado

In [18]:
# Verifica si existen montos negativos o  0 
df_pagos_tra[df_pagos_tra.monto_capital_programado <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


## monto_interes_programado

In [19]:
# Verifica si existen montos negativos o  0 
df_pagos_tra[df_pagos_tra.monto_interes_programado <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


## monto_pagado_total

In [20]:
# Verifica si existen montos negativos o  0 
# Resultados Esperados: Tabla Vacia
df_pagos_tra[df_pagos_tra.monto_pagado_total <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


## monto_capital_pagado

In [21]:
# Verifica si existen montos negativos o  0 
# Resultados Esperados: Tabla Vacia
df_pagos_tra[df_pagos_tra.monto_capital_pagado <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


## monto_interes_pagado

In [22]:
# Verifica si existen montos negativos o  0 
# Resultados Esperados: Tabla Vacia
df_pagos_tra[df_pagos_tra.monto_interes_pagado <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


## monto_mora_pagado

In [23]:
# Verifica si existen montos negativos o  0 
# Resultados Esperados: Tabla Vacia
df_pagos_tra[df_pagos_tra.monto_mora_pagado < 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


## saldo_capital_despues_pago

In [24]:
# Verifica si existen saldos negativos 0
# Resultados Esperados: Tabla Vacia
df_pagos_tra[df_pagos_tra.saldo_capital_despues_pago < 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


## dias_retraso

In [25]:
# Verifica si existen saldos negativos 0
df_pagos_tra[df_pagos_tra.dias_retraso < 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


## estado_pago

In [26]:
# Verifica el formato del texto 
# Resultados Esperados: 'Puntual', 'Muy Tardío', 'Con Gracia', 'Tardío'
df_pagos_tra.estado_pago.unique()

array(['Puntual', 'Con Gracia', 'Tardío', 'Muy Tardío'], dtype=object)

## canal_pago

In [27]:
# Verifica el formato del texto 
# Resultados Esperados: 'APP Móvil', 'Débito Automático', 'Agencia', 'Transferencia Interbancaria', 'Cajero ATM',      'Web Bancaria', 'Agente Bancario', n/a'

df_pagos_tra.canal_pago.unique()

array(['APP Móvil', 'Débito Automático', 'Agencia',
       'Transferencia Interbancaria', 'Cajero ATM', 'Web Bancaria',
       'Agente Bancario', 'n/a'], dtype=object)

## referencia_pago

In [28]:
# Verifica si existen valores con espacios vacios innecesarios o que no tengas 13 caracteres 
# Resultados Esperados: Tabla Vacia
df_pagos_tra.referencia_pago[
    (df_pagos_tra.referencia_pago
        != df_pagos_tra.referencia_pago.str.strip().str.upper())
    |
    (df_pagos_tra.referencia_pago.apply(len)!= 13)
]

Series([], Name: referencia_pago, dtype: object)

In [29]:
# Verifica el formato de la referencia del pago REF00000000(PAGO_ID) => 13 caracteres 
# Resultados Esperados: Tabla Vacia
canal_pag_gene = (
    "REF"
    + df_pagos_tra["pago_id"]
        .astype(str)
        .str.zfill(10)
)

df_pagos_tra[df_pagos_tra.referencia_pago != canal_pag_gene]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


# Segunda verificacion de la calidad de columnas y tablas

## Secuencialidad numero_cuota V2

In [30]:
# Verifica que existan registros secuenciales, es decir si tenemos como maximo el numero de cuotas pagadas 20, existan cuotas del 1 al 20 
# Es similar a la funcion de ventana ROW_NUMBER() en SQL server. 

# Ordena por prestamo_id y numero de cuota, no se usa fecha_pago, porque no esta limpia
verificar_secuencia = df_pagos_tra.copy()
verificar_secuencia = verificar_secuencia.sort_values(by=['prestamo_id','numero_cuota'],ascending=True)
verificar_secuencia['row_number'] = (
    verificar_secuencia
    .groupby(by='prestamo_id')
    .cumcount() + 1
)

# Verifica si existen registros donde el numero de cuota no coincida con el numero de registro de ese prestamo
# Resultados Esperados: Tabla Vacia
verificar_secuencia[['pago_id','prestamo_id','numero_cuota','row_number']][
    verificar_secuencia.numero_cuota != verificar_secuencia.row_number
]

,pago_id,prestamo_id,numero_cuota,row_number


## Verificar fecha_vencimiento_cuota V2

In [31]:
# fecha de vencimiento para la cuota numero 1 
df_group_fecha_min = df_pagos_tra[['pago_id','prestamo_id','fecha_vencimiento_cuota']][df_pagos_tra.numero_cuota == 1].copy()

df_fecha_venci = df_prestamos_tra[['prestamo_id','fecha_primer_pago_programado']].copy()

# LEFT JOIN, por prestamo_id
df_merge_fecha_min_fecha_venci = df_group_fecha_min.merge(
    right=df_fecha_venci,
    on='prestamo_id',
    how='left'
)

# Verifica que la fecha_vencimiento sea igual a la fecha_primer_pago_programado establecido en prestamos
# Resultados Esperados: Tabla Vacia
df_merge_fecha_min_fecha_venci[
    df_merge_fecha_min_fecha_venci.fecha_vencimiento_cuota 
        != df_merge_fecha_min_fecha_venci.fecha_primer_pago_programado
]

,pago_id,prestamo_id,fecha_vencimiento_cuota,fecha_primer_pago_programado


In [32]:

pagos_ordenados = (
    df_pagos_tra[['pago_id','prestamo_id','numero_cuota','fecha_vencimiento_cuota']]
    .sort_values(by=['prestamo_id','numero_cuota'])
    .copy()
)
# Selecciona el valor del registro anterior, similar a LAG() en SQL
pagos_ordenados['fecha_anterior'] = (
    pagos_ordenados
    .groupby('prestamo_id')['fecha_vencimiento_cuota']
    .shift(1)
)
# Muestra las fechas que no siguen la logica de sumar 30 dias por cada nuevo pago
# Resultados Esperados: Tabla Vacia  
# fecha_vencimiento_cuota_N = fecha_vencimiento_cuota_N-1 + 30
pagos_ordenados[
    (pagos_ordenados.fecha_vencimiento_cuota 
        != pagos_ordenados.fecha_anterior+pd.DateOffset(days=30))
    & 
    (pagos_ordenados.fecha_anterior.notna())
]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_anterior


In [33]:
pagos_ordenados = (
    df_pagos_tra[['pago_id','prestamo_id','numero_cuota','fecha_vencimiento_cuota']]
    .sort_values(by=['prestamo_id','numero_cuota'])
    .copy()
)
# Selecciona el valor del registro anterior, similar a LAG() en SQL
pagos_ordenados['fecha_anterior'] = (
    pagos_ordenados
    .groupby('prestamo_id')['fecha_vencimiento_cuota']
    .shift(1)
)
# Muestra las fechas que no siguen la logica de sumar 30 dias por cada nuevo pago 
# Resultados Esperados: Tabla Vacia 
pagos_ordenados[
    (pagos_ordenados.fecha_vencimiento_cuota 
        != pagos_ordenados.fecha_anterior+pd.DateOffset(days=30))
    & 
    (pagos_ordenados.fecha_anterior.notna())
]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_anterior


In [34]:
# Muestra las fechas de vencimiento futuras 
# Resultados Esperados: Tabla Vacia 
df_pagos_tra[df_pagos_tra.fecha_vencimiento_cuota.dt.date >= date.today()]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


## fecha_pago y dias_retraso

In [35]:
# Columnas a utilizar 
verificar_dias_retraso = df_pagos_tra[['pago_id','prestamo_id','fecha_vencimiento_cuota','monto_mora_pagado','fecha_pago','dias_retraso']].copy()

verificar_dias_retraso['dias_retraso_recal'] = verificar_dias_retraso.apply(
    axis=1,
    func= lambda x: 
        (x['fecha_pago'] - x['fecha_vencimiento_cuota']).days
        if x['fecha_pago'] > x['fecha_vencimiento_cuota']
        else 0
)
# Muestra los registros donde el recalculo de los dias de retraso es diferente a los dias de retraso original 
# Resultados Esperados: Tabla Vacia 
verificar_dias_retraso[
    verificar_dias_retraso.dias_retraso != verificar_dias_retraso.dias_retraso_recal
]

,pago_id,prestamo_id,fecha_vencimiento_cuota,monto_mora_pagado,fecha_pago,dias_retraso,dias_retraso_recal


## monto_cuota_programada V2

**Nota**: Ya se verifico que esta cuota sea la misma a la cuota establecida en la tabla_bronce.prestamo  
[cuota_programada](#!monto_cuota_programada)

In [36]:
# Verifica que se cumpla cuota = interes + amortizacion 
df_pagos_tra[df_pagos_tra.monto_cuota_programada 
            != round(df_pagos_tra.monto_capital_programado + df_pagos_tra.monto_interes_programado,2)]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


## monto_capital_programado y monto_interes_programado 

In [37]:
# Columnas a utilizar 
df_prestamos_veri_montos = df_prestamos_tra[['prestamo_id','tasa_interes_efectiva_anual','numero_cuotas_total','monto_original']].copy()
# Calcula el el interes mensual para cada prestamo
df_prestamos_veri_montos['tasa_efectiva_mensual'] = df_prestamos_veri_montos.tasa_interes_efectiva_anual.apply(
    lambda x: ((1+x/100)**(1/12)-1)*100
)
#-----------------------------------------------------------------------------------------------------------------
# selecciona las columnas a utilizar 
df_verificar_interes_pro = df_pagos_tra[['pago_id','prestamo_id','numero_cuota','monto_cuota_programada','monto_capital_programado','monto_interes_programado']].copy()

# LEFT JOIN, prestamo_id 
df_merge_veri_inte_pro = df_verificar_interes_pro.merge(
    right=df_prestamos_veri_montos,
    on='prestamo_id',
    how='left'
)

df_merge_veri_inte_pro.sort_values(['prestamo_id','numero_cuota'],inplace=True)

In [38]:
# Crea nuevas columnas para validar, considerando que monto capital programado es correcto
df_merge_veri_inte_pro['acumulado_capital'] = df_merge_veri_inte_pro.groupby(['prestamo_id'])['monto_capital_programado'].cumsum()

# Columnas para verificar 
df_merge_veri_inte_pro['saldo'] = df_merge_veri_inte_pro.apply(
    axis = 1,
    func = lambda x: round(x['monto_original'] - x['acumulado_capital'],2)
)

df_merge_veri_inte_pro['saldo_para_interes'] = df_merge_veri_inte_pro.groupby(['prestamo_id'])['saldo'].shift(1)

df_merge_veri_inte_pro['interes_recal'] = df_merge_veri_inte_pro.apply(
    axis = 1,
    func = lambda x: recalcular_monto_interes_programado(x['monto_original'] ,x['numero_cuota'],x['tasa_efectiva_mensual'] ,x['numero_cuotas_total'], x['monto_cuota_programada'] , x['saldo_para_interes'] )
)
# Para esta verificacion vamos a considerar un margen de +- 0.04 por errores en el redondeo
# Regresa todos los valores que no se encuentre en el rango de +- 0.04
df_merge_veri_inte_pro[~(df_merge_veri_inte_pro.monto_interes_programado.between(
    left = df_merge_veri_inte_pro.interes_recal - 0.04,
    right= df_merge_veri_inte_pro.interes_recal + 0.04,
    inclusive='both')
    )
]

,pago_id,prestamo_id,numero_cuota,monto_cuota_programada,monto_capital_programado,monto_interes_programado,tasa_interes_efectiva_anual,numero_cuotas_total,monto_original,tasa_efectiva_mensual,acumulado_capital,saldo,saldo_para_interes,interes_recal


## saldo_capital_despues_pago V2

In [ ]:
# Verifica que el saldo calculado sea igual al saldo_capital_despues_pago
df_verificar_saldo_capital_despues_pago = df_pagos_tra[['pago_id','prestamo_id','numero_cuota','saldo_capital_despues_pago']].sort_values(['prestamo_id','numero_cuota']).copy()
df_verificar_saldo_capital_despues_pago[df_verificar_saldo_capital_despues_pago.saldo_capital_despues_pago != df_merge_veri_inte_pro.saldo]

,pago_id,prestamo_id,numero_cuota,saldo_capital_despues_pago


In [40]:
#--------------------------------------------------------------------------------------------------
# Verifca si el saldo capital_vigente de prestamos es igual al saldo_capital_despues_pago de pagos 
#--------------------------------------------------------------------------------------------------
# Obtiene saldo capital vigente despues de pagar la ultima cuota de cada prestamo
idx_cuota_max = df_verificar_saldo_capital_despues_pago.groupby('prestamo_id').agg(
    idx_num_cuota_max = ('numero_cuota','idxmax')
).reset_index()
df_verificar_saldo_capital_despues_pago = df_verificar_saldo_capital_despues_pago.loc[idx_cuota_max.idx_num_cuota_max]

df_prestamos_veri_saldo_capi = df_prestamos_tra[['prestamo_id','saldo_capital_vigente']].copy() 

# LEFT JOIN, prestamo_id 
df_merge_veri_saldo_capital = df_verificar_saldo_capital_despues_pago.merge(
    right=df_prestamos_veri_saldo_capi,
    on='prestamo_id',
    how='left'
)

# Muestra los registros donde no coinciden los valores 
# El cambio se va a realizar en 09_validacion_pagos_prestamos 
df_merge_veri_saldo_capital[df_merge_veri_saldo_capital.saldo_capital_despues_pago != df_merge_veri_saldo_capital.saldo_capital_vigente]

,pago_id,prestamo_id,numero_cuota,saldo_capital_despues_pago,saldo_capital_vigente


## monto_capital_pagado V2

In [41]:
# verifica si el monto pagado no coincide con el monto programado
df_pagos_tra[df_pagos_tra.monto_capital_pagado != df_pagos_tra.monto_capital_programado]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


## monto_interes_pagado V2

In [42]:
# verifica si el monto pagado no coincide con el monto programado
df_pagos_tra[df_pagos_tra.monto_interes_pagado != df_pagos_tra.monto_interes_programado]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


## monto_pagado_total V2

**Nota**: Como vemos que capital e interes pagado coinciden con los programados los podemos considerar columnas ancla, entonces ahora lo que tendríamos que revisar el monto_mora_pagado o monto_pagado_total.

In [43]:
# Muestra las columnas donde no coincida el monto pagado con el monto capital, el monto intereses y el monto de mora
df_pagos_total_incorrecto = df_pagos_tra[
    df_pagos_tra.monto_pagado_total.round(2) 
        != (df_pagos_tra.monto_capital_pagado + df_pagos_tra.monto_interes_pagado + df_pagos_tra.monto_mora_pagado).round(2)
]
df_pagos_total_incorrecto

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


## Revision fecha_pago, dias_retraso y monto_mora_pagado

In [ ]:
# Verifica si el monto mora es correcto para los registros con errores 
# De prestamos 
df_prestamos_veri_mora = df_prestamos_tra[['prestamo_id','tasa_interes_efectiva_anual','numero_cuotas_total','monto_original']].copy()

# Calcula el interes diario para cada prestamo 
df_prestamos_veri_mora['tasa_efectiva_diaria'] = df_prestamos_veri_mora.tasa_interes_efectiva_anual.apply(
    lambda x: ((1+x/100)**(1/360)-1)*100
)
#-------------------------------------------------------------------------------------------------------------------
# recalculo de monto_mora_pagado 
veri_monto_mora = df_pagos_tra[['pago_id','prestamo_id','numero_cuota','fecha_vencimiento_cuota','monto_capital_programado','monto_pagado_total','monto_capital_pagado','monto_interes_pagado','monto_mora_pagado','fecha_pago','dias_retraso']].copy()

df_merge_monto_mora = df_prestamos_veri_mora.merge(
    right=veri_monto_mora,
    how='right',
    on='prestamo_id'
)
# Columnas a utilizar 
veri_monto_mora.sort_values(['prestamo_id','numero_cuota'],inplace=True)
df_merge_monto_mora['acumulado_capital'] = df_merge_monto_mora.groupby(['prestamo_id'])['monto_capital_programado'].cumsum()
df_merge_monto_mora['saldo'] = df_merge_monto_mora.apply(
    axis = 1,
    func = lambda x: round(x['monto_original'] - x['acumulado_capital'],2)
) 
# Selecciona el saldo para el interes usando los registros anteriores, similar a LAG(), en sql
df_merge_monto_mora['saldo_para_interes'] = df_merge_monto_mora.groupby(['prestamo_id'])['saldo'].shift(1)

df_merge_monto_mora['monto_mora_recal'] = df_merge_monto_mora.apply(
    axis = 1,
    func= lambda x:  recalcular_monto_mora_pagado(x['numero_cuota'],x['saldo_para_interes'],x['monto_original'], x['tasa_efectiva_diaria'],x['dias_retraso'])
)
# Muestra los registros donde no coindia el monto_mora_pagado con monto_mora_recal 
# Resultados Esperados: 24,552 registros 
"""
Para la IA o la persona que lea esto, este fragmento, puede que genere dudas, porque, son muchos registros, la lógica es la siguiente: 
Lo que pasa es que dias_retraso se recalcula de monto_mora_pagado, entonces muchas veces salen días decimales como 4.3 o 5.6 días, como evidentemente esto no tiene sentido, se redondea, esto genera 4 y 6 siguiendo con el ejemplo, lo que pasa es que ahora al recalcular se usan los valores redondeados 4 y 6, entonces la mora no sale exacta, porque, con el redondeo el recalculo de mora es menor o mayor al monto mora original. 
Entonces esto genera otra duda, ¿El margen de error es aceptable?, bueno a ver técnicamente los margenes de error no son muy grandes algunos si son de 20 o 10 soles, pero si se observa el VA que se utiliza para calcular la mora son números grandes, entonces variaciones pequeñas en decimales generan margenes mas grandes. 
No estoy seguro si fue la mejor opción, pero bueno es la opción que me parece más logíca, con suerte en un futuro cuando vea esto identifique mejores soluciones :).
"""
df_pagos_mora_inco = df_merge_monto_mora[['pago_id','prestamo_id','tasa_efectiva_diaria','monto_pagado_total','monto_capital_pagado','monto_interes_pagado','saldo_para_interes','dias_retraso','monto_mora_pagado','monto_mora_recal']][
    ~(df_merge_monto_mora.monto_mora_pagado.between(
    left = df_merge_monto_mora.monto_mora_recal - 0.04,
    right= df_merge_monto_mora.monto_mora_recal + 0.04,
    inclusive='both'))
    ]
df_pagos_mora_inco

,pago_id,prestamo_id,tasa_efectiva_diaria,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,saldo_para_interes,dias_retraso,monto_mora_pagado,monto_mora_recal
9,10,1,0.055495,410.16,277.52,116.11,6918.39,4,16.53,15.37
44,45,2,0.059375,236.09,132.42,93.50,5204.16,3,10.17,9.28
80,81,4,0.115677,688.52,542.35,125.47,NaN,5,20.70,20.61
96,97,5,0.088304,438.55,241.80,168.83,6291.81,5,27.92,27.83
111,112,5,0.088304,421.31,359.71,50.92,1897.55,6,10.68,10.08
...,...,...,...,...,...,...,...,...,...,...
134880,134881,6493,0.028190,3486.76,184.15,3220.89,379300.58,1,81.72,106.92
134884,134885,6493,0.028190,3621.26,190.48,3214.56,378554.55,2,216.22,213.46
134890,134891,6493,0.028190,3514.00,200.39,3204.65,377387.13,1,108.96,106.39
134894,134895,6494,0.074278,897.29,834.54,18.80,834.54,69,43.95,43.87


In [45]:
# Verifica los errores en dias retraso 
verificar_dias_retraso = df_pagos_tra[['pago_id','prestamo_id','fecha_vencimiento_cuota','monto_mora_pagado','fecha_pago','dias_retraso']].copy()

verificar_dias_retraso['dias_retraso_recal'] = verificar_dias_retraso.apply(
    axis=1,
    func= lambda x: 
        (x['fecha_pago'] - x['fecha_vencimiento_cuota']).days
        if x['fecha_pago'] > x['fecha_vencimiento_cuota']
        else 0
)

verificar_dias_retraso[
    verificar_dias_retraso.dias_retraso != verificar_dias_retraso.dias_retraso_recal
]

,pago_id,prestamo_id,fecha_vencimiento_cuota,monto_mora_pagado,fecha_pago,dias_retraso,dias_retraso_recal


In [46]:
# Verifica si en los registros con monto_mora incorrectos se encuentras los pagos_totales_incorrectos 
df_pagos_mora_inco.loc[df_pagos_mora_inco.pago_id.isin(df_pagos_total_incorrecto.pago_id)]

,pago_id,prestamo_id,tasa_efectiva_diaria,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,saldo_para_interes,dias_retraso,monto_mora_pagado,monto_mora_recal


## monto_mora_pagado V2

In [47]:
# monto_mora_pagado = monto_pagado_total -  monto_capital_pagado - monto_interes_pagado
df_pagos_tra['monto_mora_pagado'] = df_pagos_tra.apply(
    axis = 1,
    func = lambda x: abs(round(x['monto_pagado_total'] - x['monto_capital_pagado'] - x['monto_interes_pagado'],2))
)

df_pagos_tra[df_pagos_tra.monto_pagado_total.round(2) != (df_pagos_tra.monto_capital_pagado + df_pagos_tra.monto_interes_pagado + df_pagos_tra.monto_mora_pagado).round(2)]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


## dias_retraso V2

**Formula para calcular el periodo (dias)**:  
 ln(1+ I/VA) / ln(1 + I)

In [48]:
# calculamos los días de retraso en funcion de monto_mora_pagado 
df_verificar_retraso = df_pagos_tra[['pago_id','prestamo_id','monto_capital_pagado','monto_mora_pagado','saldo_capital_despues_pago','dias_retraso']].copy()
df_merge_veri_dias_retraso = df_verificar_retraso.merge(
    right=df_prestamos_veri_mora,
    on='prestamo_id',
    how='left'
)

df_merge_veri_dias_retraso['saldo_para_interes'] = df_merge_veri_dias_retraso.apply(
    axis = 1,
    func = lambda x: 
    round(
        x['saldo_capital_despues_pago'] + x['monto_capital_pagado'],
        2
    )
)
# Recalcula los dias de retraso, en este caso realizo un redondeo sin decimales, porque, no existen dias decimales
df_merge_veri_dias_retraso['recal_dias_retraso'] = df_merge_veri_dias_retraso.apply(
    axis = 1,
    func = lambda x: 
    round(
        np.log(1+(x['monto_mora_pagado']/x['saldo_para_interes'])) / np.log(1 + x['tasa_efectiva_diaria']/100)
    ) 
)
# Resultados esperados: 1 registro, este registro se genera porque en el calculo sale algo aprox 0.49.. días entonces redondea directo a 0, en lugar de 1 a pesar de tener un monto mora pagado
df_merge_veri_dias_retraso[df_merge_veri_dias_retraso.dias_retraso != df_merge_veri_dias_retraso.recal_dias_retraso]

,pago_id,prestamo_id,monto_capital_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,tasa_interes_efectiva_anual,numero_cuotas_total,monto_original,tasa_efectiva_diaria,saldo_para_interes,recal_dias_retraso
130171,130172,6245,219.89,70.03,478270.09,1,11.3289,342,479350.12,0.029815,478489.98,0


In [49]:
# Algunas verificaciones 
dias_retra_sin_pago = df_pagos_tra[(df_pagos_tra.dias_retraso != 0) & (df_pagos_tra.monto_mora_pagado == 0)].shape[0]
sin_dias_retra_con_pago = df_pagos_tra[(df_pagos_tra.dias_retraso == 0) & (df_pagos_tra.monto_mora_pagado != 0)].shape[0]
print(f"Registros con dias de retraso y sin pago por mora: {dias_retra_sin_pago}")
print(f"Registros sin dias de retraso y con pago por mora: {sin_dias_retra_con_pago}")

Registros con dias de retraso y sin pago por mora: 0
Registros sin dias de retraso y con pago por mora: 0


In [50]:
#============================================================================================
# Verifica si los dias de retraso del ultimo pago coinciden con los dias de mora de prestamos 
#============================================================================================
# Seleccionamos algunas columnas a utilizar 
df_veri_dias_retraso = df_pagos_tra[['pago_id','prestamo_id','dias_retraso','numero_cuota','fecha_pago']].copy()
df_prestamos_veri_dias_retraso = df_prestamos_tra[['prestamo_id','numero_cuotas_total','dias_mora']].copy()

# Obtenemos el indice de la ultima cuota  
df_veri_dias_retraso = df_veri_dias_retraso.groupby('prestamo_id').agg(
    idx_cuota_maxima = ('numero_cuota','idxmax'),
).reset_index()
# obtemos los valores asociados a estos indices 
df_veri_dias_retraso = df_pagos_tra.loc[df_veri_dias_retraso.idx_cuota_maxima,['pago_id','prestamo_id','dias_retraso','numero_cuota','fecha_pago']].copy()
# LEFT JOIN, prestamo_id
df_merge_veri_dias_retraso = df_veri_dias_retraso.merge(
    right = df_prestamos_veri_dias_retraso,
    on='prestamo_id',
    how='left'
)
# Muestras los registros donde los dias de retraso no coinciden con los dias de mora, del ultimo pago 
# Resultados Esperados: Tabla Vacia
df_merge_veri_dias_retraso[df_merge_veri_dias_retraso.dias_retraso != df_merge_veri_dias_retraso.dias_mora]

,pago_id,prestamo_id,dias_retraso,numero_cuota,fecha_pago,numero_cuotas_total,dias_mora


## fecha_pago V2

In [51]:
df_pagos_tra['fecha_pago'] = df_pagos_tra.apply(
    axis = 1,
    func = lambda x: 
        x['fecha_vencimiento_cuota'] + pd.DateOffset(days=x['dias_retraso'])
)
# Muestra los registros donde los dias de retraso, no sean iguales a la diferencia en dias de fecha pago y fecha vencimiento cuota 
# Resultados Esperados: Tabla Vacia
df_pagos_tra[df_pagos_tra.dias_retraso != (df_pagos_tra.fecha_pago - df_pagos_tra.fecha_vencimiento_cuota).dt.days ]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


In [52]:
# Muestra las fechas de pago que son futura
# Resultados Esperados: Tabla Vacia
df_pagos_tra[df_pagos_tra.fecha_pago.dt.date >= date.today()]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago,dwh_fecha_carga


In [53]:
#--------------------------------------------------------------------------------------------
# Verifica que la ultima fecha_pago de pagos, sea igual a fecha_ultimo_pago_real de prestamos
#--------------------------------------------------------------------------------------------
df_verificar_fechas_pag = df_pagos_tra[['pago_id','prestamo_id','numero_cuota','fecha_pago']].copy()
df_prestamos_veri_fechas = df_prestamos_tra[['prestamo_id','numero_cuotas_total','numero_cuotas_pagadas','numero_cuotas_pendientes','fecha_ultimo_pago_real']].copy()

# Obtenemos la ultima cuota y la ultima fecha de pago 
df_indice_ultima_cuota = df_verificar_fechas_pag.groupby('prestamo_id').agg(
    idx_max = ('numero_cuota','idxmax')
).reset_index()
df_verificar_fechas_pag = df_verificar_fechas_pag.loc[df_indice_ultima_cuota.idx_max]
# LEFT JOIN, prestamo_id
df_merge_veri_fechas = df_verificar_fechas_pag.merge(
    right = df_prestamos_veri_fechas,
    on='prestamo_id',
    how='left'
)
# Muestras los registros con fechas donde el ultimo pago no coincide 
# # Resultados Esperados: Tabla Vacia
df_merge_veri_fechas[df_merge_veri_fechas.fecha_pago != df_merge_veri_fechas.fecha_ultimo_pago_real]

,pago_id,prestamo_id,numero_cuota,fecha_pago,numero_cuotas_total,numero_cuotas_pagadas,numero_cuotas_pendientes,fecha_ultimo_pago_real


## estado_pago V2

**Puntual**: 0  
**Con Gracia**: 1 - 8  
**Tardío**: 9 - 30  
**Muy Tardío**: 31 - 150  

In [54]:
# Arreglamos el estado_pago segun la logica de negocio
df_pagos_tra['estado_pago'] = df_pagos_tra.dias_retraso.apply(formato_estado_pago)
df_pagos_tra.estado_pago.value_counts()

estado_pago
Puntual       109555
Con Gracia     20967
Tardío          3545
Muy Tardío       844
Name: count, dtype: int64